# DFF Paper Experiments (Max 5 Cells, Ultra-Critical)



This notebook is designed to be **complete and auditable** in the sense that every reported number is tied to an executable definition.



## What this notebook reproduces

- Loads paired KITTI–VKITTI2 manifests from `controlnet_*_dataset/metadata.jsonl` (schema: `image`, `conditioning_image`, `prompt`).

- Loads your trained ControlNet checkpoints from `out_*` and generates images conditioned on `conditioning_image`.

- Computes **IV** (LPIPS if available, else pixel MSE fallback with explicit flag).

- Computes **OV** for Steering (and optionally YOLOP DA/LL if its repo+weights are accessible) using frozen SUT heads.

- Computes **DFF** using a **CF-XAI-style “minimal deletion” optimizer** (Fong & Vedaldi–style meaningful perturbation) that searches for a small mask whose deletion changes the SUT output. This is the closest self-contained approximation to mask-based counterfactual decisive maps in this workspace.



## Ultra-critical paper issues (forced into explicit choices)

1) **Regression “flip” ambiguity (steering):** the paper says “flip output”, but steering is continuous. Here we define a flip as an output change larger than a configurable $\tau$ (default `STEERING_FLIP_TAU_RADIANS`).

2) **Calibration method inconsistency:** the paper alternates between “calibrator predicts knobs” and “update $\Theta$ via backprop”. This notebook evaluates the *saved artifacts*; it does not retrain/calibrate.

3) **Thresholds:** this notebook implements $\varepsilon_{90}$/$\varepsilon_{95}$ as 90/95th percentiles of DFF distances on the calibration split, per SUT.

4) **Metric signs:** we report both `ov_loss` and `ov_sim` where possible.



## Scope constraints

- Exact CF-XAI “mask-and-infill from learned prior” is not included in this workspace; we use deletion against a blur baseline.

- YOLOP evaluation requires the YOLOP repo root + weights; the notebook tries to auto-detect them from `run_03_*.sh`/`run_04_*.sh`, and otherwise skips YOLOP cleanly.


In [ ]:
from __future__ import annotations



import json

import math

import re

from pathlib import Path

from typing import Any, Dict, List, Optional, Tuple



import numpy as np





# ----------------------------

# Configuration (edit here)

# ----------------------------

ROOT = Path.cwd()  # workspace root (vkitti_seg_final7)



# Choose ONE dataset manifest to drive the experiment (all have same schema)

DATASET_DIR = ROOT / "controlnet_vkitti_ovf_steering_corrected_dataset"

MANIFEST = DATASET_DIR / "metadata.jsonl"



# Trained ControlNet checkpoints (main, non-debug runs)

OUT_DIRS: Dict[str, Path] = {

    "baseline": ROOT / "out_01_baseline",

    "ovf_steering": ROOT / "out_02_ovf_steering_beta0p70",

    "ovf_yolop_da": ROOT / "out_03_ovf_yolop_da_beta0p50",

    "ovf_yolop_ll": ROOT / "out_04_ovf_yolop_ll_beta0p45",

    "dff_steering": ROOT / "out_05_dff_steering_beta0p35_lam0p10",

    "dff_yolop_da": ROOT / "out_06_dff_yolop_da_beta0p30_lam0p08",

    "dff_yolop_ll": ROOT / "out_07_dff_yolop_ll_beta0p30_lam0p08",

}



# Diffusion base

BASE_SDXL = "stabilityai/stable-diffusion-xl-base-1.0"



# Training used 1248x376 (KITTI-like). Keep consistent.

RESOLUTION: Tuple[int, int] = (1248, 376)  # (W, H)



# Small by default; raise if you want full split.

N_CAL_GEN = 8

N_TEST_GEN = 8



SEED = 123

NUM_INFERENCE_STEPS = 30

GUIDANCE_SCALE = 7.0

CONTROLNET_CONDITIONING_SCALE = 1.0



# Caches

CACHE_DIR = ROOT / "notebook_cache"

GEN_DIR = CACHE_DIR / "generated"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

GEN_DIR.mkdir(parents=True, exist_ok=True)



# ----------------------------

# SUT paths (auto-detected from run_*.sh when possible)

# ----------------------------

STEERING_ARCH_PATH = Path("/networkhome/WMGDS/safaei_d/3_generators/models/architectures/For_steering_angle_prediction_tweaked_v3.py")

STEERING_WEIGHTS_PATH = Path("/networkhome/WMGDS/safaei_d/3_generators/models/weights/final_model.pth")



# YOLOP repo/weights are NOT in this workspace; we try to infer paths from run scripts.

YOLOP_REPO_ROOT: Optional[Path] = None

YOLOP_WEIGHTS_PATH: Optional[Path] = None



# ----------------------------

# DFF / CF-XAI hyperparameters (keep modest; CF optimization is expensive)

# ----------------------------

K_CF = 6

CF_STEPS = 25

CF_LR = 0.05

CF_DOWNSAMPLE = 32

CF_LAMBDA_AREA = 0.20

CF_LAMBDA_TV = 0.005



# Steering-specific: define what a counterfactual "flip" means for regression

STEERING_FLIP_TAU_RADIANS = 0.10





def _require(cond: bool, msg: str) -> None:

    if not cond:

        raise RuntimeError(msg)





def _read_text(path: Path) -> str:

    return path.read_text(encoding="utf-8")





def _try_autodetect_sut_paths() -> None:

    global STEERING_ARCH_PATH, STEERING_WEIGHTS_PATH, YOLOP_REPO_ROOT, YOLOP_WEIGHTS_PATH



    steer_run = ROOT / "run_02_ovf_steering.sh"

    if steer_run.exists():

        txt = _read_text(steer_run)

        m_arch = re.search(r"--steering_arch_path\s+(\S+)", txt)

        m_w = re.search(r"--steering_weights_path\s+(\S+)", txt)

        if m_arch:

            STEERING_ARCH_PATH = Path(m_arch.group(1))

        if m_w:

            STEERING_WEIGHTS_PATH = Path(m_w.group(1))



    for run_name in [

        "run_03_ovf_yolop_da.sh",

        "run_04_ovf_yolop_ll.sh",

        "run_06_dff_yolop_da.sh",

        "run_07_dff_yolop_ll.sh",

    ]:

        p = ROOT / run_name

        if not p.exists():

            continue

        txt = _read_text(p)

        m_root = re.search(r"--yolop_root\s+(\S+)", txt) or re.search(r"--yolop_repo_root\s+(\S+)", txt)

        m_w = re.search(r"--yolop_weights_path\s+(\S+)", txt)

        if m_root and YOLOP_REPO_ROOT is None:

            YOLOP_REPO_ROOT = Path(m_root.group(1))

        if m_w and YOLOP_WEIGHTS_PATH is None:

            YOLOP_WEIGHTS_PATH = Path(m_w.group(1))





_try_autodetect_sut_paths()



print("ROOT=", ROOT)

print("MANIFEST=", MANIFEST)

print("STEERING_ARCH_PATH=", STEERING_ARCH_PATH)

print("STEERING_WEIGHTS_PATH=", STEERING_WEIGHTS_PATH)

print("YOLOP_REPO_ROOT=", YOLOP_REPO_ROOT)

print("YOLOP_WEIGHTS_PATH=", YOLOP_WEIGHTS_PATH)



_require(MANIFEST.exists(), f"Missing manifest: {MANIFEST}")

for name, out_dir in OUT_DIRS.items():

    _require((out_dir / "config.json").exists(), f"Missing config.json in {out_dir}")

    _require((out_dir / "diffusion_pytorch_model.safetensors").exists(), f"Missing safetensors in {out_dir}")

print("OK: found manifest + all out_* checkpoints")


In [ ]:
# ----------------------------
# Load paired data + scene-wise split (paper claims split by scene)
# ----------------------------


def _read_jsonl(path: Path) -> List[Dict[str, Any]]:
    items: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            items.append(json.loads(line))
    return items


def _scene_from_conditioning_path(p: str) -> str:
    parts = Path(p).parts
    for token in parts:
        if token.startswith("Scene"):
            return token
    return "UNKNOWN"


items = _read_jsonl(MANIFEST)
_require(len(items) in (2126, 2127), f"Unexpected item count: {len(items)} (expected ~2126)")

for it in items[:10]:
    _require(set(it.keys()) >= {"image", "conditioning_image", "prompt"}, f"Bad schema: {it.keys()}")
    _require(Path(it["image"]).exists(), f"Missing real image on disk: {it['image']}")
    _require(Path(it["conditioning_image"]).exists(), f"Missing conditioning image on disk: {it['conditioning_image']}")

scenes = sorted({ _scene_from_conditioning_path(it["conditioning_image"]) for it in items })
print("Scenes found:", scenes)

# Paper says: 5 scenes; 80/20 by scene -> hold out 1 scene.
# Ultra-critical: paper does not specify which scene is held out.
HELDOUT_SCENES = [scenes[-1]]

calib = [it for it in items if _scene_from_conditioning_path(it["conditioning_image"]) not in HELDOUT_SCENES]
heldout = [it for it in items if _scene_from_conditioning_path(it["conditioning_image"]) in HELDOUT_SCENES]

print(f"HELDOUT_SCENES={HELDOUT_SCENES}")
print(f"calib={len(calib)} heldout={len(heldout)} total={len(items)}")

# ---------------------------------------------------------
# Sampling with Forced Pilot Frame Inclusion (Paper Table III)
# ---------------------------------------------------------
PILOT_SCENE = "Scene20"
PILOT_FRAME_ID = "00750"  # Look for this frame index

def _find_pilot_index(dataset: List[Dict[str, Any]]) -> Optional[int]:
    for i, item in enumerate(dataset):
        # Check if path contains the pilot frame ID
        # e.g. .../Scene20/clone/frames/rgb/Camera_0/rgb_00750.jpg
        if PILOT_FRAME_ID in item["image"] and PILOT_SCENE in item["conditioning_image"]:
            return i
    return None

rng = np.random.default_rng(SEED)

# Calibration split sampling
calib_idx = rng.choice(len(calib), size=min(N_CAL_GEN, len(calib)), replace=False)
calib_subset = [calib[int(i)] for i in calib_idx]

# Held-out split sampling (Force pilot frame)
heldout_indices = list(range(len(heldout)))
pilot_idx = _find_pilot_index(heldout)

if pilot_idx is not None:
    print(f"✅ Found Pilot Frame ({PILOT_SCENE}, {PILOT_FRAME_ID}) at index {pilot_idx} in heldout.")
    # Ensure pilot is in the selection
    if pilot_idx not in heldout_indices:
        # Should not happen if we select from range, but good for logic
        heldout_indices.append(pilot_idx)
    
    # Select N-1 random others, then add pilot
    other_indices = [i for i in heldout_indices if i != pilot_idx]
    selected_others = rng.choice(other_indices, size=min(N_TEST_GEN - 1, len(other_indices)), replace=False)
    final_heldout_idx = np.concatenate([selected_others, [pilot_idx]])
else:
    print(f"⚠️ Pilot Frame ({PILOT_SCENE}, {PILOT_FRAME_ID}) NOT found in heldout. Using random selection.")
    final_heldout_idx = rng.choice(len(heldout), size=min(N_TEST_GEN, len(heldout)), replace=False)

heldout_subset = [heldout[int(i)] for i in final_heldout_idx]

print("Example calib item:", calib_subset[0])
print("Example heldout item:", heldout_subset[0])

In [ ]:
# ----------------------------

# Generate images using each trained ControlNet (small subset)

# ----------------------------



from PIL import Image



import importlib

import torch

from typing import Any



try:

    _diffusers = importlib.import_module("diffusers")

    ControlNetModel = getattr(_diffusers, "ControlNetModel")

    StableDiffusionXLControlNetPipeline = getattr(_diffusers, "StableDiffusionXLControlNetPipeline")

except Exception as e:

    raise RuntimeError(

        "diffusers is required. Install: pip install diffusers transformers accelerate safetensors"

    ) from e





device = "cuda" if torch.cuda.is_available() else "cpu"

dtype = torch.bfloat16 if (device == "cuda") else torch.float32

print("device=", device, "dtype=", dtype)





def _load_rgb(path: str) -> Image.Image:

    return Image.open(path).convert("RGB")





@torch.inference_mode()

def _generate_one(pipe: Any, item: Dict[str, Any], seed: int) -> Image.Image:

    cond = _load_rgb(item["conditioning_image"])

    g = torch.Generator(device=device).manual_seed(int(seed))

    out = pipe(

        prompt=item["prompt"],

        image=cond,

        generator=g,

        num_inference_steps=int(NUM_INFERENCE_STEPS),

        guidance_scale=float(GUIDANCE_SCALE),

        controlnet_conditioning_scale=float(CONTROLNET_CONDITIONING_SCALE),

        width=int(RESOLUTION[0]),

        height=int(RESOLUTION[1]),

    )

    return out.images[0]





def _make_pipe(controlnet_dir: Path) -> Any:

    # Ultra-critical: we load the *fine-tuned* ControlNet from out_dir.

    controlnet = ControlNetModel.from_pretrained(controlnet_dir, torch_dtype=dtype)

    pipe = StableDiffusionXLControlNetPipeline.from_pretrained(

        BASE_SDXL,

        controlnet=controlnet,

        torch_dtype=dtype,

        variant=None,

        use_safetensors=True,

    )

    pipe.to(device)

    pipe.set_progress_bar_config(disable=True)

    return pipe





def _ensure_dir(p: Path) -> None:

    p.mkdir(parents=True, exist_ok=True)





def _variant_cache_dir(variant: str, split_name: str) -> Path:

    return CACHE_DIR / "generated" / variant / split_name





def _item_cache_path(variant: str, split_name: str, idx: int) -> Path:

    return _variant_cache_dir(variant, split_name) / f"{idx:06d}.png"





generated_index: Dict[str, Dict[str, List[Path]]] = {"calib": {}, "heldout": {}}



for split_name, subset in [("calib", calib_subset), ("heldout", heldout_subset)]:

    for variant, out_dir in OUT_DIRS.items():

        cache_dir = _variant_cache_dir(variant, split_name)

        _ensure_dir(cache_dir)

        paths: List[Path] = []

        for i, item in enumerate(subset):

            p = _item_cache_path(variant, split_name, i)

            paths.append(p)

        generated_index[split_name][variant] = paths



print("Prepared cache index.")





def _need_generation(paths: List[Path]) -> bool:

    return any((not p.exists()) for p in paths)





for split_name, subset in [("calib", calib_subset), ("heldout", heldout_subset)]:

    for variant, out_dir in OUT_DIRS.items():

        paths = generated_index[split_name][variant]

        if not _need_generation(paths):

            print("Cache hit:", variant, split_name)

            continue

        print("Generating:", variant, split_name)

        pipe = _make_pipe(out_dir)

        for i, item in enumerate(subset):

            p = paths[i]

            if p.exists():

                continue

            seed = int(SEED + 1000 * (0 if split_name == "calib" else 1) + 17 * i)

            img = _generate_one(pipe, item, seed)

            img.save(p)

        del pipe

        if torch.cuda.is_available():

            torch.cuda.empty_cache()



print("Generation done (or cache already present).")


In [ ]:
def _lpips_distance(a: Image.Image, b: Image.Image, device: str) -> tuple[float, str]:
    """Returns (distance, metric_name)."""
    # Ensure dimensions match by resizing a to match b
    if a.size != b.size:
        a = a.resize(b.size, Image.BICUBIC)
        
    try:
        import lpips  # type: ignore
        loss_fn = lpips.LPIPS(net="alex").to(device)
        ta = _to_torch_image(a, device) * 2 - 1
        tb = _to_torch_image(b, device) * 2 - 1
        with torch.inference_mode():
            d = loss_fn(ta, tb)
        return float(d.item()), "lpips"
    except Exception:
        ta = _to_torch_image(a, device)
        tb = _to_torch_image(b, device)
        return float(tF.mse_loss(ta, tb).item()), "pixel_mse_fallback"

def _iv_score(iv_dist: float, iv_metric: str) -> float:
    # Paper uses 1-LPIPS (clamped) only for visualization.
    if iv_metric != "lpips":
        return float("nan")
    return float(1.0 - min(max(iv_dist, 0.0), 1.0))

def _iou_binary(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    inter = (a * b).sum(dim=(1, 2, 3))
    union = ((a + b) > 0).float().sum(dim=(1, 2, 3))
    return inter / (union + eps)

def _tv_loss(m: torch.Tensor) -> torch.Tensor:
    dh = (m[:, :, 1:, :] - m[:, :, :-1, :]).abs().mean()
    dw = (m[:, :, :, 1:] - m[:, :, :, :-1]).abs().mean()
    return dh + dw

def _blur_baseline(x: torch.Tensor) -> torch.Tensor:
    k = 21
    pad = k // 2
    return tF.avg_pool2d(tF.pad(x, (pad, pad, pad, pad), mode="reflect"), kernel_size=k, stride=1)

@torch.enable_grad()
def cf_decisive_map_min_deletion(
    model: torch.nn.Module,
    x: torch.Tensor,
    mode: str,
    tau: float,
    steps: int,
    lr: float,
    downsample: int,
    lambda_area: float,
    lambda_tv: float,
    seed: int,
) -> torch.Tensor:
    """CF-XAI-style decisive map via minimal deletion."""
    torch.manual_seed(int(seed))
    np.random.seed(int(seed) % (2**32 - 1))

    model.eval()
    with torch.no_grad():
        y0 = model(x)

    H, W = x.shape[-2], x.shape[-1]
    mh = max(1, H // downsample)
    mw = max(1, W // downsample)

    m_logits = torch.randn((1, 1, mh, mw), device=x.device, dtype=torch.float32, requires_grad=True)
    opt = torch.optim.Adam([m_logits], lr=float(lr))
    x0 = _blur_baseline(x)

    for _ in range(int(steps)):
        opt.zero_grad(set_to_none=True)
        m = torch.sigmoid(m_logits)
        m_up = tF.interpolate(m, size=(H, W), mode="bilinear", align_corners=False)
        x_cf = m_up * x + (1.0 - m_up) * x0
        y_cf = model(x_cf)

        if mode == "steering":
            delta = (y_cf - y0).abs().mean()
            change_loss = tF.relu(torch.tensor(float(tau), device=x.device) - delta)
        elif mode == "segmentation":
            def _as_mask(logits: torch.Tensor) -> torch.Tensor:
                if logits.dim() == 4 and logits.shape[1] > 1:
                    logits = logits[:, :1]
                return (torch.sigmoid(logits) > 0.5).float()
            m0 = _as_mask(y0)
            m1 = _as_mask(y_cf)
            iou = _iou_binary(m0, m1).mean()
            delta = 1.0 - iou
            change_loss = tF.relu(torch.tensor(float(tau), device=x.device) - delta)
        else:
            raise ValueError(f"Unknown mode: {mode}")

        area_loss = (1.0 - m).mean()
        tv = _tv_loss(m)
        loss = change_loss + float(lambda_area) * area_loss + float(lambda_tv) * tv
        loss.backward()
        opt.step()

    with torch.no_grad():
        m = torch.sigmoid(m_logits)
        m_up = tF.interpolate(m, size=(H, W), mode="bilinear", align_corners=False)
        decisive = (1.0 - m_up).clamp(0.0, 1.0)
        pooled = tF.adaptive_avg_pool2d(decisive, (16, 16)).squeeze(0).squeeze(0)
    return pooled

def dff_distance_cf(
    model: torch.nn.Module,
    x_real: torch.Tensor,
    x_syn: torch.Tensor,
    mode: str,
    tau: float,
    k_cf: int,
    seed0: int,
) -> float:
    maps_r: List[torch.Tensor] = []
    maps_s: List[torch.Tensor] = []
    for k in range(int(k_cf)):
        seed = int(seed0 + 100 * k)
        maps_r.append(cf_decisive_map_min_deletion(model, x_real, mode, tau, CF_STEPS, CF_LR, CF_DOWNSAMPLE, CF_LAMBDA_AREA, CF_LAMBDA_TV, seed))
        maps_s.append(cf_decisive_map_min_deletion(model, x_syn, mode, tau, CF_STEPS, CF_LR, CF_DOWNSAMPLE, CF_LAMBDA_AREA, CF_LAMBDA_TV, seed))
    mr = torch.stack(maps_r, dim=0).mean(dim=0)
    ms = torch.stack(maps_s, dim=0).mean(dim=0)
    return float(((mr - ms) ** 2).mean().item())

def _load_steering_sut() -> Optional[torch.nn.Module]:
    if not (STEERING_ARCH_PATH.exists() and STEERING_WEIGHTS_PATH.exists()):
        print("Steering SUT not found; skipping steering OV/DFF.")
        return None
    import importlib.util
    spec = importlib.util.spec_from_file_location("steering_arch", str(STEERING_ARCH_PATH))
    _require(spec is not None and spec.loader is not None, "Failed to load steering architecture module spec")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)  # type: ignore
    candidate_names = ["AutonomousDrivingModel", "Net", "PilotNet", "Model"]
    cls = None
    for name in candidate_names:
        if hasattr(mod, name):
            cls = getattr(mod, name)
            break
    if cls is None:
        raise RuntimeError(f"Could not find a steering model class in {STEERING_ARCH_PATH}. Tried: {candidate_names}")
    model = cls()
    ckpt = torch.load(STEERING_WEIGHTS_PATH, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    model.load_state_dict(ckpt, strict=False)
    model.eval().to(device)
    for p in model.parameters():
        p.requires_grad_(False)
    return model

def _try_load_yolop() -> Tuple[Optional[torch.nn.Module], Optional[torch.nn.Module]]:
    if YOLOP_REPO_ROOT is None or YOLOP_WEIGHTS_PATH is None:
        print("YOLOP paths not detected; skipping YOLOP OV/DFF.")
        return None, None
    if not YOLOP_REPO_ROOT.exists() or not YOLOP_WEIGHTS_PATH.exists():
        print("YOLOP paths invalid on disk; skipping.")
        return None, None
    sys.path.insert(0, str(YOLOP_REPO_ROOT))
    try:
        from lib.models import get_net  # type: ignore
    except Exception as e:
        print("Failed to import YOLOP from repo root; skipping.")
        return None, None
    yolop = get_net(False)
    ckpt = torch.load(YOLOP_WEIGHTS_PATH, map_location="cpu")
    if isinstance(ckpt, dict) and "state_dict" in ckpt:
        ckpt = ckpt["state_dict"]
    yolop.load_state_dict(ckpt, strict=False)
    yolop.eval().to(device)
    for p in yolop.parameters():
        p.requires_grad_(False)
    def da_head(x: torch.Tensor) -> torch.Tensor:
        out = yolop(x)
        if isinstance(out, (list, tuple)) and len(out) >= 3:
            return out[1]
        raise RuntimeError("Unexpected YOLOP output structure")
    def ll_head(x: torch.Tensor) -> torch.Tensor:
        out = yolop(x)
        if isinstance(out, (list, tuple)) and len(out) >= 3:
            return out[2]
        raise RuntimeError("Unexpected YOLOP output structure")
    class _Wrap(torch.nn.Module):
        def __init__(self, f):
            super().__init__()
            self.f = f
        def forward(self, x):
            return self.f(x)
    return _Wrap(da_head).to(device), _Wrap(ll_head).to(device)

steer_sut = _load_steering_sut()
yolop_da_sut, yolop_ll_sut = _try_load_yolop()

def _steer_scalar(y: torch.Tensor) -> torch.Tensor:
    return y.reshape(y.shape[0], -1).mean(dim=1)

def _steering_ov(y_real: torch.Tensor, y_gen: torch.Tensor) -> Tuple[float, float]:
    y_real = _steer_scalar(y_real)
    y_gen = _steer_scalar(y_gen)
    delta = (y_gen - y_real).abs().mean().item()
    return float(delta), float(math.exp(-5.0 * float(delta)))

def _compute_rows(split_name: str, split_items: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for j, item in enumerate(split_items):
        real = _load_rgb(item["image"])
        # FIX: Resize real image to match generated resolution to prevent LPIPS size mismatch
        real = real.resize(RESOLUTION, Image.BICUBIC)

        x_real = _to_torch_image(real, device=device)
        x_real_y = _to_yolop_input(real, device=device)

        for variant, gen_paths in generated_index[split_name].items():
            gen = Image.open(gen_paths[j]).convert("RGB")
            x_gen = _to_torch_image(gen, device=device)
            x_gen_y = _to_yolop_input(gen, device=device)

            iv_dist, iv_name = _lpips_distance(real, gen, device=device)
            row: Dict[str, Any] = {
                "split": split_name,
                "idx": j,
                "variant": variant,
                "scene": _scene_from_conditioning_path(item["conditioning_image"]),
                "image_path": item["image"], # Store path to identify Pilot frame
                "iv_dist": float(iv_dist),
                "iv_metric": iv_name,
                "iv_score": _iv_score(float(iv_dist), iv_name),
            }

            if steer_sut is not None:
                with torch.no_grad():
                    y_r = steer_sut(x_real)
                    y_g = steer_sut(x_gen)
                ov_loss, ov_sim = _steering_ov(y_r, y_g)
                row.update({"steer_ov_loss": ov_loss, "steer_ov_sim": ov_sim})
                row["steer_dff"] = dff_distance_cf(steer_sut, x_real, x_gen, "steering", float(STEERING_FLIP_TAU_RADIANS), int(K_CF_PAPER), int(SEED + 10000 + 17 * j))

            if yolop_da_sut is not None:
                with torch.no_grad():
                    y_r = yolop_da_sut(x_real_y)
                    y_g = yolop_da_sut(x_gen_y)
                m_r = (torch.sigmoid(y_r[:, :1]) > 0.5).float()
                m_g = (torch.sigmoid(y_g[:, :1]) > 0.5).float()
                iou = float(_iou_binary(m_r, m_g).mean().item())
                row.update({"da_ov_loss": float(1.0 - iou), "da_ov_sim": iou})
                row["da_dff"] = dff_distance_cf(yolop_da_sut, x_real_y, x_gen_y, "segmentation", 0.15, int(K_CF_PAPER), int(SEED + 20000 + 19 * j))

            if yolop_ll_sut is not None:
                with torch.no_grad():
                    y_r = yolop_ll_sut(x_real_y)
                    y_g = yolop_ll_sut(x_gen_y)
                m_r = (torch.sigmoid(y_r[:, :1]) > 0.5).float()
                m_g = (torch.sigmoid(y_g[:, :1]) > 0.5).float()
                iou = float(_iou_binary(m_r, m_g).mean().item())
                row.update({"ll_ov_loss": float(1.0 - iou), "ll_ov_sim": iou})
                row["ll_dff"] = dff_distance_cf(yolop_ll_sut, x_real_y, x_gen_y, "segmentation", 0.15, int(K_CF_PAPER), int(SEED + 30000 + 23 * j))

            rows.append(row)
    return rows

In [ ]:
# ----------------------------
# Compute Metrics for All Generated Images
# ----------------------------
print("Computing metrics... this may take a while depending on K_CF.")
rows = _compute_rows("calib", calib_subset) + _compute_rows("heldout", heldout_subset)
df = pd.DataFrame(rows)

# Save raw results
out_csv = CACHE_DIR / "metrics_results_paper.csv"
df.to_csv(out_csv, index=False)
print("Saved raw metrics to:", out_csv)
display(df.head())

In [ ]:
# ----------------------------
# Paper Tables Generation
# ----------------------------

# --- Table II: Calibrated DFF Thresholds ---
print("\n=== Table II: Calibrated DFF Thresholds (Calibration Split) ===")
def _quantiles(series: pd.Series) -> Tuple[float, float]:
    v = series.dropna().to_numpy()
    if len(v) == 0: return float("nan"), float("nan")
    return float(np.quantile(v, 0.90)), float(np.quantile(v, 0.95))

thresholds: Dict[str, Dict[str, float]] = {}
table_ii_data = []

for key, label in [("steer_dff", "Steering"), ("da_dff", "YOLOP-DA"), ("ll_dff", "YOLOP-LL")]:
    if key not in df.columns: continue
    # Compute on CALIB split, BASELINE variant
    base_cal = df[(df.split == "calib") & (df.variant == "baseline")][key]
    eps90, eps95 = _quantiles(base_cal)
    thresholds[key] = {"eps90": eps90, "eps95": eps95}
    table_ii_data.append({"SUT": label, "eps90": eps90, "eps95": eps95})

df_table_ii = pd.DataFrame(table_ii_data)
display(df_table_ii)
print(df_table_ii.to_latex(index=False, float_format="%.3f"))


# --- Table IV: O1 Coupling (Correlations) ---
print("\n=== Table IV: O1 Coupling (Held-out, Baseline) ===")
def _spearman_ci(a: pd.Series, b: pd.Series) -> Tuple[float, Tuple[float, float]]:
    try:
        res = spearmanr(a, b)
        r = res.correlation
        # Fisher transformation for CI
        z = np.arctanh(r)
        se = 1.0 / np.sqrt(len(a) - 3)
        z_low, z_high = z - 1.96 * se, z + 1.96 * se
        return float(r), (float(np.tanh(z_low)), float(np.tanh(z_high)))
    except Exception:
        return float("nan"), (float("nan"), float("nan"))

table_iv_data = []
base_h = df[(df.split == "heldout") & (df.variant == "baseline")]

# Steering
if "steer_ov_sim" in base_h.columns and "steer_dff" in base_h.columns:
    r_iv_ov, ci_iv_ov = _spearman_ci(base_h["iv_dist"], base_h["steer_ov_sim"])
    r_ov_dff, ci_ov_dff = _spearman_ci(base_h["steer_ov_sim"], base_h["steer_dff"])
    table_iv_data.append({"SUT": "Steer", "rho(IV,OV)": r_iv_ov, "CI_IV_OV": ci_iv_ov, "rho(OV,DFF)": r_ov_dff, "CI_OV_DFF": ci_ov_dff})

# DA
if "da_ov_sim" in base_h.columns and "da_dff" in base_h.columns:
    r_iv_ov, ci_iv_ov = _spearman_ci(base_h["iv_dist"], base_h["da_ov_sim"])
    r_ov_dff, ci_ov_dff = _spearman_ci(base_h["da_ov_sim"], base_h["da_dff"])
    table_iv_data.append({"SUT": "DA", "rho(IV,OV)": r_iv_ov, "CI_IV_OV": ci_iv_ov, "rho(OV,DFF)": r_ov_dff, "CI_OV_DFF": ci_ov_dff})

# LL
if "ll_ov_sim" in base_h.columns and "ll_dff" in base_h.columns:
    r_iv_ov, ci_iv_ov = _spearman_ci(base_h["iv_dist"], base_h["ll_ov_sim"])
    r_ov_dff, ci_ov_dff = _spearman_ci(base_h["ll_ov_sim"], base_h["ll_dff"])
    table_iv_data.append({"SUT": "LL", "rho(IV,OV)": r_iv_ov, "CI_IV_OV": ci_iv_ov, "rho(OV,DFF)": r_ov_dff, "CI_OV_DFF": ci_ov_dff})

df_table_iv = pd.DataFrame(table_iv_data)
display(df_table_iv)


# --- Table V: O2 Core Effects (Deltas) ---
print("\n=== Table V: O2 Core Effects (Held-out, Delta = DFF_Calibrated - Baseline) ===")
# We compare DFF-calibrated variant vs Baseline.
# DFF variants: "dff_steering", "dff_yolop_da", "dff_yolop_ll"
# Metrics: IV Score (higher better), OV Sim (higher better), DFF Dist (lower better)

table_v_data = []
h = df[df.split == "heldout"]

# Helper to get mean of a metric for a variant
def _get_mean(variant, metric):
    return h[h.variant == variant][metric].mean()

# Steering
if "dff_steering" in OUT_DIRS:
    base_iv = _get_mean("baseline", "iv_score")
    base_ov = _get_mean("baseline", "steer_ov_sim")
    base_dff = _get_mean("baseline", "steer_dff")
    
    cal_iv = _get_mean("dff_steering", "iv_score")
    cal_ov = _get_mean("dff_steering", "steer_ov_sim")
    cal_dff = _get_mean("dff_steering", "steer_dff")
    
    table_v_data.append({
        "SUT": "Steer",
        "Delta_IV": cal_iv - base_iv,
        "Delta_OV": cal_ov - base_ov,
        "Delta_DFF": cal_dff - base_dff
    })

# DA
if "dff_yolop_da" in OUT_DIRS:
    base_iv = _get_mean("baseline", "iv_score")
    base_ov = _get_mean("baseline", "da_ov_sim")
    base_dff = _get_mean("baseline", "da_dff")
    
    cal_iv = _get_mean("dff_yolop_da", "iv_score")
    cal_ov = _get_mean("dff_yolop_da", "da_ov_sim")
    cal_dff = _get_mean("dff_yolop_da", "da_dff")
    
    table_v_data.append({
        "SUT": "DA",
        "Delta_IV": cal_iv - base_iv,
        "Delta_OV": cal_ov - base_ov,
        "Delta_DFF": cal_dff - base_dff
    })

# LL
if "dff_yolop_ll" in OUT_DIRS:
    base_iv = _get_mean("baseline", "iv_score")
    base_ov = _get_mean("baseline", "ll_ov_sim")
    base_dff = _get_mean("baseline", "ll_dff")
    
    cal_iv = _get_mean("dff_yolop_ll", "iv_score")
    cal_ov = _get_mean("dff_yolop_ll", "ll_ov_sim")
    cal_dff = _get_mean("dff_yolop_ll", "ll_dff")
    
    table_v_data.append({
        "SUT": "LL",
        "Delta_IV": cal_iv - base_iv,
        "Delta_OV": cal_ov - base_ov,
        "Delta_DFF": cal_dff - base_dff
    })

df_table_v = pd.DataFrame(table_v_data)
display(df_table_v)
print(df_table_v.to_latex(index=False, float_format="%+.3f"))

In [ ]:
# ----------------------------
# Table III: Pilot Test (Specific Frame)
# ----------------------------
print("\n=== Table III: Pilot Test (Specific Frame) ===")

# Find the pilot frame in the dataframe
# We look for PILOT_FRAME_ID in image_path
pilot_rows = df[df["image_path"].astype(str).str.contains(PILOT_FRAME_ID)]

if len(pilot_rows) == 0:
    print(f"⚠️ Pilot frame {PILOT_FRAME_ID} not found in generated results. Showing first available frame instead.")
    pilot_idx_val = df.iloc[0]["idx"]
    pilot_rows = df[df["idx"] == pilot_idx_val]
else:
    print(f"✅ Found Pilot frame {PILOT_FRAME_ID}.")

# We need rows for: Baseline, OVF-X, DFF-X for each SUT.
# Table columns: SUT, Variant, LPIPS, OV loss, DFF
table_iii_data = []

# Helper to extract metrics for a specific variant from the pilot rows
def _get_pilot_metrics(sut_label, variant_key, ov_loss_col, dff_col):
    row = pilot_rows[pilot_rows["variant"] == variant_key]
    if len(row) == 0: return None
    # LPIPS is iv_dist (lower is better)
    lpips = row.iloc[0]["iv_dist"]
    ov_loss = row.iloc[0][ov_loss_col]
    dff = row.iloc[0][dff_col]
    return {
        "SUT": sut_label,
        "Variant": variant_key,
        "LPIPS": lpips,
        "OV loss": ov_loss,
        "DFF": dff
    }

# Steering
# Variants: baseline, ovf_steering, dff_steering
for v, label in [("baseline", "Baseline"), ("ovf_steering", "OVF-Steer"), ("dff_steering", "DFF-Steer")]:
    m = _get_pilot_metrics("Steer", v, "steer_ov_loss", "steer_dff")
    if m: table_iii_data.append(m)

# DA
# Variants: baseline, ovf_yolop_da, dff_yolop_da
for v, label in [("baseline", "Baseline"), ("ovf_yolop_da", "OVF-DA"), ("dff_yolop_da", "DFF-DA")]:
    m = _get_pilot_metrics("DA", v, "da_ov_loss", "da_dff")
    if m: table_iii_data.append(m)

# LL
# Variants: baseline, ovf_yolop_ll, dff_yolop_ll
for v, label in [("baseline", "Baseline"), ("ovf_yolop_ll", "OVF-LL"), ("dff_yolop_ll", "DFF-LL")]:
    m = _get_pilot_metrics("LL", v, "ll_ov_loss", "ll_dff")
    if m: table_iii_data.append(m)

df_table_iii = pd.DataFrame(table_iii_data)
display(df_table_iii)
print(df_table_iii.to_latex(index=False, float_format="%.3f"))


# ----------------------------
# Figures
# ----------------------------

# --- Figure 2: DFF Distributions & CDFs ---
print("\n=== Figure 2: DFF Distributions & CDFs ===")
try:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # (a) Distribution (KDE)
    ax = axes[0]
    colors = {"steer_dff": "blue", "da_dff": "orange", "ll_dff": "green"}
    labels = {"steer_dff": "Steering", "da_dff": "YOLOP-DA", "ll_dff": "YOLOP-LL"}
    
    for key in ["steer_dff", "da_dff", "ll_dff"]:
        if key not in df.columns: continue
        vals = df[(df.split == "heldout") & (df.variant == "baseline")][key].dropna()
        sns.kdeplot(vals, ax=ax, label=labels[key], color=colors[key], fill=True, alpha=0.1)
        # Add vertical line for eps95
        eps95 = thresholds[key]["eps95"]
        ax.axvline(eps95, color=colors[key], linestyle="--", alpha=0.7)
    
    ax.set_title("(a) Distribution of DFF distances")
    ax.set_xlabel("DFF Distance")
    ax.legend()
    
    # (b) CDF
    ax = axes[1]
    for key in ["steer_dff", "da_dff", "ll_dff"]:
        if key not in df.columns: continue
        vals = df[(df.split == "heldout") & (df.variant == "baseline")][key].dropna().sort_values()
        p = np.linspace(0, 1, len(vals))
        ax.plot(vals, p, label=labels[key], color=colors[key])
        
        # Mark 95% crossing
        eps95 = thresholds[key]["eps95"]
        ax.plot(eps95, 0.95, marker="o", color=colors[key])
        
    ax.axhline(0.95, color="gray", linestyle="--", label="95%")
    ax.set_title("(b) Empirical CDFs P(D <= eps)")
    ax.set_xlabel("DFF Distance")
    ax.set_ylabel("Probability")
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    fig.savefig(CACHE_DIR / "fig2_dff_distance.pdf")
    print("Saved Fig 2 to:", CACHE_DIR / "fig2_dff_distance.pdf")

except Exception as e:
    print(f"Plotting failed: {e}")


# --- Figure 3: Qualitative Comparison (Grid) ---
print("\n=== Figure 3: Qualitative Comparison (Pilot Frame) ===")
# We want to show: Real, Baseline, OVF-Steer, OVF-DA, OVF-LL, DFF-Steer, DFF-DA, DFF-LL
# For the pilot frame.

if len(pilot_rows) > 0:
    # Get the index of the pilot frame in the generated lists
    # We need to find the 'idx' value from the dataframe
    pilot_idx_val = pilot_rows.iloc[0]["idx"]
    split_name = pilot_rows.iloc[0]["split"]
    
    # Load images
    # Real
    real_path = pilot_rows.iloc[0]["image_path"]
    real_img = Image.open(real_path).convert("RGB")
    
    # Generated
    def _load_gen(variant):
        p = generated_index[split_name][variant][pilot_idx_val]
        return Image.open(p).convert("RGB")
    
    imgs = {
        "Real": real_img,
        "Baseline": _load_gen("baseline"),
        "OVF-Steer": _load_gen("ovf_steering"),
        "OVF-DA": _load_gen("ovf_yolop_da"),
        "OVF-LL": _load_gen("ovf_yolop_ll"),
        "DFF-Steer": _load_gen("dff_steering"),
        "DFF-DA": _load_gen("dff_yolop_da"),
        "DFF-LL": _load_gen("dff_yolop_ll"),
    }
    
    # Plot grid 2x4
    fig, axes = plt.subplots(2, 4, figsize=(20, 8))
    axes = axes.flatten()
    
    order = ["Real", "Baseline", "OVF-Steer", "OVF-DA", "OVF-LL", "DFF-Steer", "DFF-DA", "DFF-LL"]
    
    for i, key in enumerate(order):
        if key in imgs:
            axes[i].imshow(imgs[key])
            axes[i].set_title(key)
            axes[i].axis("off")
            
            # Add metrics text if available
            if key != "Real":
                # Find row
                var_key = key.lower().replace("-", "_") # e.g. OVF-Steer -> ovf_steering
                if key == "Baseline": var_key = "baseline"
                
                row = pilot_rows[pilot_rows["variant"] == var_key]
                if len(row) > 0:
                    iv = row.iloc[0]["iv_score"]
                    # Determine which OV/DFF to show based on variant type
                    ov_txt = ""
                    dff_txt = ""
                    if "steer" in var_key or var_key=="baseline":
                        ov_txt += f"S_OV:{row.iloc[0]['steer_ov_sim']:.2f} "
                        dff_txt += f"S_D:{row.iloc[0]['steer_dff']:.2f} "
                    if "da" in var_key:
                        ov_txt += f"DA_OV:{row.iloc[0]['da_ov_sim']:.2f} "
                        dff_txt += f"DA_D:{row.iloc[0]['da_dff']:.2f} "
                    if "ll" in var_key:
                        ov_txt += f"LL_OV:{row.iloc[0]['ll_ov_sim']:.2f} "
                        dff_txt += f"LL_D:{row.iloc[0]['ll_dff']:.2f} "
                        
                    axes[i].text(10, 30, f"IV:{iv:.2f}\n{ov_txt}\n{dff_txt}", 
                                 color="white", fontsize=10, bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.show()
    fig.savefig(CACHE_DIR / "fig3_qualitative.pdf")
    print("Saved Fig 3 to:", CACHE_DIR / "fig3_qualitative.pdf")
else:
    print("Skipping Fig 3 (no pilot rows found).")